In [1]:
user_id = "tp99965073"
domain = "AXISB.COM"
app_name = "Fraud_&_AML"
yarn_queue = "root.DEV.AML_FRAUD_CS"

EXECUTOR_MEMORY=28
EXECUTOR_CORE=4

INITIAL_EXECUTORS=8
MAX_EXECUTORS=50     # 16x4 = 64 cores; lower this to your approved cap if smaller (YARN grants only what is free)
MIN_EXECUTORS=2

In [2]:
import sys
from pyspark.sql import SparkSession, SQLContext
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark.sql.types as T
from datetime import datetime, date,timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import time
import numpy as np


from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))


# Construct the keytab path and the principal
keytab_path = f"/home/{user_id}@axisb.com/{user_id}.keytab"
principal = f"{user_id}@{domain}"

# Execute the commands
!kdestroy
!kinit -k -t "{keytab_path}" {principal}
try:
    spark.stop()
    print("Existing Spark Session Killed")
except:
    print("")
print("Creating new spark session since: ", datetime.now())
spark = SparkSession\
    .builder\
    .appName(f'JH_{user_id}_{app_name}')\
    .config('spark.sql.execution.arrow.enabled', "true")\
    .config('spark.master','yarn')\
    .config('spark.deploy-mode', 'cluster')\
    .config("spark.principal",f"{user_id}@AXISB.COM") \
    .config("spark.keytab",f"{user_id}.keytab")\
    .config('spark.dynamicAllocation.enabled','true') \
    .config('spark.dynamicAllocation.minExecutors',MIN_EXECUTORS) \
    .config('spark.dynamicAllocation.initialExecutors',INITIAL_EXECUTORS) \
    .config('spark.dynamicAllocation.maxExecutors',MAX_EXECUTORS) \
    .config('spark.executor.cores',EXECUTOR_CORE) \
    .config('spark.dynamicAllocation.executorAllocationRatio','0.8')\
    .config('spark.executor.memory',f'{EXECUTOR_MEMORY}g')\
    .config('spark.driver.memory','16g')\
    .config('spark.driver.cores','1')\
    .config('spark.executor.memoryOverhead','8g')\
    .config('spark.sql.shuffle.partitions','12000')\
    .config('spark.shuffle.service.enabled','true')\
    .config('spark.network.timeout','600s')\
    .config('spark.shuffle.io.maxRetries','10')\
    .config('spark.shuffle.io.retryWait','15s')\
    .config('spark.stage.maxConsecutiveAttempts','8')\
    .config('spark.maxRemoteBlockSizeFetchToMem','512m')\
    .config("spark.sql.broadcastTimeout", "-1")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")\
    .config("spark.yarn.queue", yarn_queue)\
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')\
    .config('spark.hadoop.hive.exec.dynamic.partition.mode', 'nonstrict')\
    .config('spark.sql.parquet.output.committer.class', 'org.apache.parquet.hadoop.ParquetOutputCommitter')\
    .config('spark.sql.sources.commitProtocolClass', 'org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol')\
    .config("spark.jars", "hdfs:///jars/iceberg-spark-runtime-3.3_2.12-1.6.1.jar,/opt/cloudera/parcels/CDH/jars/ojdbc8.jar")\
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')\
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')\
    .config('spark.sql.catalog.spark_catalog.type', 'hive')\
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')\
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')\
    .config('spark.sql.adaptive.autoBroadcastJoinThreshold','50MB')\
    .config("spark.sql.legacy.allowNonEmptyLocationInCTAS", "true")\
    .enableHiveSupport()\
    .getOrCreate()
sc = SparkContext.getOrCreate()
print("New Spark Session created at : ", datetime.now())

new_log_level = "ERROR"
sc.setLogLevel(new_log_level)
print(f"Logs level set to {new_log_level}")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


Creating new spark session since:  2026-07-06 23:38:35.466801


26/07/06 23:38:36 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/07/06 23:38:36 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 23:38:55 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/07/06 23:38:55 WARN  spark.SparkConf: [Thread-10]: The configuration key 'spark.maxRemoteBlockSizeFetc

New Spark Session created at :  2026-07-06 23:39:07.014980
Logs level set to ERROR


----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 44860)
Traceback (most recent call last):
  File "/data1/miniconda3/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/data1/miniconda3/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/data1/miniconda3/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/data1/miniconda3/lib/python3.8/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/cloudera/parcels/SPARK3-3.3.2.3.3.7190.10-1-1.p0.59394740/lib/spark3/python/pyspark/accumulators.py", line 282, in handle
    poll(accum_updates)
  File "/opt/cloudera/parcels/SPARK3-3.3.2.3.3.7190.10-1-1.p0.59394740/lib/spark3/python/pyspark/accumulators.py", line 253, in poll
    if func():
  File "/opt/cloud

# Notebook 03 (v2) — seed-anchored multi-hop subgraph (iterative joins)

Extracts, per anchor month (Sep/Oct/Nov 2025), the seed-anchored **k-hop** neighbourhood from the
full v2 transaction graph (02b) and integer-indexes it for igraph (notebook 04).

* **Seeds** = accounts of HISTORICAL-bad PL customers (bad loan disbursed BEFORE the anchor).
* **Candidates** = accounts of customers whose PL was disbursed IN the anchor month; `target_fraud`
  (held out) = that window loan's outcome.
* **Iterative-join BFS** (mentor's "iterative joins"): frontier = seeds ∪ candidates; expand k hops on
  the anchor's trailing-3-month TRANSFERRED_TO edges; **hubs dropped** + **degree-capped transit** so a
  popular node can't drag in all its senders. Leak-free: only trailing-3-month edges per anchor.

Outputs `subgraph_node_index` (+ idx + seed/candidate/target/attrs) and `subgraph_edges_idx`
(integer endpoints + weight) per anchor → igraph input for 04. HEAVY — run OFF-HOURS.


In [3]:
# ---- PARAMETERS ----
from pyspark.sql import functions as F
from pyspark import StorageLevel

WRITE_DB = "payment_cards_dev"; EMP = "TP99965073"
BASE_TBL = f"{WRITE_DB}.{EMP}_pl_graph_base_cohort_v1"   # nb 01
OWN_TBL  = f"{WRITE_DB}.{EMP}_v2_edges_ownership"        # nb 02 (cust -> acct)
EDGES_TXN= f"{WRITE_DB}.{EMP}_v2_edges_transferred"      # nb 02b (src, dst, txn_month, sum_amount, txn_count)
NODES_ACCT=f"{WRITE_DB}.{EMP}_v2_nodes_account"          # nb 02b (node_id, account_network_type, is_hub)

BASE_CUST, BASE_MONTH, BASE_FRAUD = "lddf_customer_id", "month_period", "final_fraud_flag"

# anchor -> trailing 3 months (leak-free)
ANCHOR_TRAILING = {
    "202509": ["202506", "202507", "202508"],
    "202510": ["202507", "202508", "202509"],
    "202511": ["202508", "202509", "202510"],
}
KHOP                 = 3        # mentor: not limited to 2 hops
EXPANSION_MAX_DEGREE = 50       # transit gate: don't expand THROUGH nodes with undirected degree > this
DROP_HUB             = True     # drop hub-beneficiary edges before BFS
MAX_NODES_WARN       = 8_000_000
# POINT-IN-TIME seed maturity buffer. The base cohort carries only DISBURSAL month + a FINAL fraud flag
# (no fraud-confirmation date), so a loan disbursed before the anchor but only CONFIRMED fraud AFTER it is
# still, by default, a seed -> mild optimism. Requiring seeds to be disbursed >= LAG months before the anchor
# approximates "already confirmed by decision time". LAG=0 = current behaviour (exact reproduction);
# set LAG=3 and rerun 03->06 to prove the lift is robust to the confirmation-lag caveat.
SEED_MATURITY_LAG    = 0        # months. 0 = seeds are ALL frauds disbursed STRICTLY BEFORE the anchor month
                                # (anchor-month & later loans are NEVER seeds -> no disbursal-timing leakage).
                                # This is the POC setting: uses the full pre-anchor seed set to demonstrate the
                                # graph signal with the scarce positives. Set 3 for the point-in-time robustness
                                # footnote (drops the freshest, possibly-not-yet-confirmed frauds).
def _ym_minus(ym, k):
    t = int(str(ym)[:4]) * 12 + (int(str(ym)[4:6]) - 1) - int(k)
    return "%04d%02d" % (t // 12, t % 12 + 1)

OUT_NIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_node_index"
OUT_EIDX = f"{WRITE_DB}.{EMP}_v2_subgraph_edges_idx"

def ACC_KEY(c): return F.upper(F.trim(F.col(c).cast("string")))
def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

base = spark.table(BASE_TBL)
own  = spark.table(OWN_TBL).select(F.col("src").alias("cust"), F.col("dst").alias("acct"))
nacc = spark.table(NODES_ACCT).select("node_id", "account_network_type", "is_hub")
print("anchors:", list(ANCHOR_TRAILING.keys()), "| KHOP:", KHOP, "| degree cap:", EXPANSION_MAX_DEGREE)


Hive Session ID = f12f70cd-fb7a-4ceb-a273-c26fcd936211


anchors: ['202509', '202510', '202511'] | KHOP: 3 | degree cap: 50


## A. Per-anchor seed / candidate ACCOUNTS (mapped from PL customers via ownership)


In [4]:
def seed_cand_accounts(anchor):
    b = base.select(ACC_KEY(BASE_CUST).alias("cust"),
                    F.col(BASE_MONTH).cast("string").alias("m"),
                    F.col(BASE_FRAUD).cast("int").alias("ff")).where(F.col("cust").isNotNull())
    # candidates: loan disbursed IN the anchor month; target = that loan's FINAL outcome (correct: the thing we predict)
    cand_cust = (b.where(F.col("m") == anchor).groupBy("cust").agg(F.max("ff").alias("target_fraud")))
    # seeds: customer had a BAD loan disbursed BEFORE the anchor, with an optional maturity buffer so we do not
    # count loans too fresh to have plausibly been confirmed by decision time (point-in-time robustness).
    seed_cutoff = _ym_minus(anchor, SEED_MATURITY_LAG)          # LAG=0 -> == anchor (current behaviour)
    seed_cust = (b.where((F.col("m") < seed_cutoff) & (F.col("ff") == 1)).select("cust").distinct())
    # map customers -> their accounts (any owned account that exists in the graph)
    cand_acc = (cand_cust.join(own, "cust").select(F.col("acct").alias("id"), "target_fraud")
                .groupBy("id").agg(F.max("target_fraud").alias("target_fraud"))
                .withColumn("is_candidate", F.lit(1)))
    seed_acc = (seed_cust.join(own, "cust").select(F.col("acct").alias("id")).distinct()
                .withColumn("is_known_bad", F.lit(1)))
    return seed_acc, cand_acc


## B. Build the seed-anchored subgraph for one anchor (TIME-RESPECTING iterative-join BFS)
Money flows forward in time: we expand a chain to the next account only if that transaction happened
on/after the account received money (txn month >= arrival month). This traces real money-flow chains and
drops the spurious static connections. Roots (seeds + candidates) start active from month 0.


In [5]:
def build_subgraph(anchor):
    months = ANCHOR_TRAILING[anchor]
    seed_acc, cand_acc = seed_cand_accounts(anchor)

    # anchor-window transfer edges, hubs dropped
    e = spark.table(EDGES_TXN).where(F.col("txn_month").isin(*months)) \
            .select("src", "dst", "sum_amount", "txn_count", F.col("txn_month").cast("int").alias("tm"))
    if DROP_HUB:
        hubs = nacc.where(F.col("is_hub") == 1).select(F.col("node_id").alias("dst"))
        e = e.join(hubs, "dst", "left_anti")                       # drop edges into hub beneficiaries
    e = e.where(F.col("src") != F.col("dst")).persist(StorageLevel.DISK_ONLY)

    # TEMPORAL substrate: directed edges keeping the transaction month (yyyyMM int) for time-respecting BFS
    de = e.select("src", "dst", "tm").distinct().persist(StorageLevel.DISK_ONLY)
    # undirected degree ONLY for the transit gate (don't expand THROUGH hubs)
    adj = (e.select("src", "dst").union(e.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
           .distinct().persist(StorageLevel.DISK_ONLY))
    udeg = adj.groupBy("src").agg(F.count("*").alias("d")).select(F.col("src").alias("id"), "d")

    roots = (seed_acc.select("id").union(cand_acc.select("id")).distinct()).persist(StorageLevel.DISK_ONLY)
    transit = (udeg.where(F.col("d") <= EXPANSION_MAX_DEGREE).select("id")
               .union(roots).distinct()).persist(StorageLevel.DISK_ONLY)

    # TIME-RESPECTING directed forward BFS: from a node reached with arrival month `arr`, follow only a
    # directed edge (src=node) with month >= arr (money can only be forwarded after it is received). Roots
    # (seeds + candidates) start at arr=0. Transit gate preserved (expand only THROUGH low-degree nodes).
    frontier = roots.withColumn("arr", F.lit(0))
    visited = roots
    for _h in range(KHOP):
        expandable = frontier.join(transit, "id", "inner")                    # (id, arr) for gate-passing nodes
        nbr = (de.join(expandable.select(F.col("id").alias("src"), "arr"), "src")
               .where(F.col("tm") >= F.col("arr"))                            # forward-in-time only
               .select(F.col("dst").alias("id"), F.col("tm").alias("arr")))
        nbr = nbr.groupBy("id").agg(F.min("arr").alias("arr"))                # earliest arrival = most permissive
        frontier = nbr.join(visited, "id", "left_anti").persist(StorageLevel.DISK_ONLY)  # first visit
        if frontier.limit(1).count() == 0:
            break
        visited = visited.union(frontier.select("id")).distinct().persist(StorageLevel.DISK_ONLY)

    reached = visited.select("id").distinct()

    # node index (contiguous 0..n-1) + attributes
    n = reached.count()
    idx = spark.createDataFrame(reached.rdd.map(lambda r: r[0]).zipWithIndex(), ["id", "idx"])
    node_index = (idx
                  .join(seed_acc, "id", "left").join(cand_acc, "id", "left")
                  .join(nacc.withColumnRenamed("node_id", "id"), "id", "left")
                  .fillna({"is_known_bad": 0, "is_candidate": 0, "target_fraud": 0, "is_hub": 0})
                  .withColumn("anchor_month", F.lit(anchor)))

    # subgraph transfer edges (both endpoints reached) -> integer endpoints + weight
    cand_edges = (e.join(idx.select(F.col("id").alias("src"), F.col("idx").alias("src_idx")), "src")
                  .join(idx.select(F.col("id").alias("dst"), F.col("idx").alias("dst_idx")), "dst")
                  .select(F.lit(anchor).alias("anchor_month"), "src_idx", "dst_idx",
                          F.log1p(F.col("sum_amount")).alias("weight"), "txn_count", "sum_amount"))

    e.unpersist(); de.unpersist(); adj.unpersist(); roots.unpersist(); transit.unpersist()
    return node_index, cand_edges, n


## C. Run all anchors, save node index + edge index


In [6]:
from functools import reduce
ni_frames, ce_frames = [], []
for a in ANCHOR_TRAILING.keys():
    print("=== building subgraph for", a, "===")
    ni, ce, n = build_subgraph(a)
    print("   subgraph nodes:", n, ("  *** > warn threshold, consider lowering KHOP/degree ***" if n > MAX_NODES_WARN else ""))
    ni_frames.append(ni); ce_frames.append(ce)

node_index = reduce(lambda x, y: x.unionByName(y), ni_frames)
cand_edges = reduce(lambda x, y: x.unionByName(y), ce_frames)
save_overwrite(node_index, OUT_NIDX)
save_overwrite(cand_edges, OUT_EIDX)
print("saved:", OUT_NIDX, "and", OUT_EIDX)


=== building subgraph for 202509 ===


26/07/06 23:55:38 ERROR scheduler.AsyncEventQueue: [dag-scheduler-event-loop]: Dropping event from queue eventLog. This likely means one of the listeners is too slow and cannot keep up with the rate at which tasks are being started by the scheduler.
                                                                                 / 12000]]]]]

   subgraph nodes: 3215394 
=== building subgraph for 202510 ===


                                                                                 / 12000]]]]]]

   subgraph nodes: 3293915 
=== building subgraph for 202511 ===


                                                                                 / 12000]]0]

   subgraph nodes: 3011792 


                                                                                286]]]000]]

saved: payment_cards_dev.TP99965073_v2_subgraph_node_index and payment_cards_dev.TP99965073_v2_subgraph_edges_idx


## D. QC — per anchor: sizes, seeds/candidates, connectivity, index contiguity


In [7]:
ni = spark.table(OUT_NIDX); ce = spark.table(OUT_EIDX)
qc_ok = True
for a in ANCHOR_TRAILING.keys():
    nia = ni.where(F.col("anchor_month") == a)
    cea = ce.where(F.col("anchor_month") == a)
    nodes = nia.count(); edges = cea.count()
    seeds = nia.where(F.col("is_known_bad") == 1).count()
    cands = nia.where(F.col("is_candidate") == 1).count()
    badc  = nia.where((F.col("is_candidate") == 1) & (F.col("target_fraud") == 1)).count()
    # candidate connectivity: candidate idx that appears on >=1 edge
    cidx = nia.where(F.col("is_candidate") == 1).select("idx")
    ep = (cea.select(F.col("src_idx").alias("idx")).union(cea.select(F.col("dst_idx").alias("idx"))).distinct())
    conn = cidx.join(ep, "idx", "inner").count()
    st = nia.agg(F.min("idx").alias("mn"), F.max("idx").alias("mx"), F.count("*").alias("n")).first()
    contiguous = (st["mn"] == 0 and st["mx"] == st["n"] - 1)
    bad_edge = cea.where((F.col("src_idx") >= nodes) | (F.col("dst_idx") >= nodes)).count()
    ok = contiguous and bad_edge == 0
    qc_ok = qc_ok and ok
    print("[%s] nodes %d | edges %d | seeds %d | candidates %d (bad %d) | cand-connected %d | idx_contig %s | bad_edge %d | OK=%s" % (
        a, nodes, edges, seeds, cands, badc, conn, contiguous, bad_edge, ok))


[202509] nodes 3215394 | edges 6071744 | seeds 14443 | candidates 70399 (bad 125) | cand-connected 56630 | idx_contig True | bad_edge 0 | OK=True


[202510] nodes 3293915 | edges 6301494 | seeds 14545 | candidates 71601 (bad 113) | cand-connected 57966 | idx_contig True | bad_edge 0 | OK=True


[202511] nodes 3011792 | edges 5670429 | seeds 14650 | candidates 64078 (bad 63) | cand-connected 51172 | idx_contig True | bad_edge 0 | OK=True


## Done


In [8]:
if qc_ok:
    print("NOTEBOOK 03 (v2) COMPLETE — seed-anchored k-hop subgraph built and integer-indexed (igraph input for 04).")
    print("Outputs:", OUT_NIDX, "(nodes + idx + is_known_bad/is_candidate/target_fraud/net_type)")
    print("        ", OUT_EIDX, "(src_idx, dst_idx, weight, txn_count -> igraph edges)")
    print("Next: notebook 04 (igraph) — PPR from is_known_bad -> score is_candidate -> validate vs target_fraud; Leiden + Infomap.")
else:
    print("NOTEBOOK 03 (v2) NOT done — a per-anchor QC failed; fix before notebook 04.")
print("Paste back the per-anchor QC line (nodes/edges/seeds/candidates/connectivity/OK).")


NOTEBOOK 03 (v2) COMPLETE — seed-anchored k-hop subgraph built and integer-indexed (igraph input for 04).
Outputs: payment_cards_dev.TP99965073_v2_subgraph_node_index (nodes + idx + is_known_bad/is_candidate/target_fraud/net_type)
         payment_cards_dev.TP99965073_v2_subgraph_edges_idx (src_idx, dst_idx, weight, txn_count -> igraph edges)
Next: notebook 04 (igraph) — PPR from is_known_bad -> score is_candidate -> validate vs target_fraud; Leiden + Infomap.
Paste back the per-anchor QC line (nodes/edges/seeds/candidates/connectivity/OK).
